# Centinela QC: entrenar el detector en Colab (GPU)

Entrena el mismo detector que `ml/train.py`, pero en la GPU de Colab. El etiquetado y el armado del dataset se hacen antes en la PC, y la evaluación y la exportación, después (ver `ml/README.md`).

**Antes de empezar**, en la PC:

```powershell
cd ml; .venv\Scripts\python autolabel.py; .venv\Scripts\python build_dataset.py
cd ..; Compress-Archive -Path ml\work\dataset -DestinationPath dataset.zip -Force
```

**En Colab:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*. Después, ejecutar las celdas en orden.

> Todo lo que está en `/content` se borra cuando se corta la sesión. Descargar `best.pt` apenas termine el entrenamiento.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi

## 2. Subir `dataset.zip`

Ejecutar la celda y elegir el archivo. Si el zip ya está en Google Drive, usar la celda siguiente.

In [ ]:
from google.colab import files

uploaded = files.upload()  # elegir dataset.zip
assert "dataset.zip" in uploaded, f"Se esperaba dataset.zip y llegó: {list(uploaded)}"

In [ ]:
# Opcional: copiar el zip desde Google Drive en lugar de subirlo.
# from google.colab import drive
# drive.mount("/content/drive")
# !cp "/content/drive/MyDrive/dataset.zip" /content/

## 3. Descomprimir y revisar

In [ ]:
!rm -rf /content/dataset
!unzip -q -o /content/dataset.zip -d /content/

from pathlib import Path

ROOT = Path("/content/dataset")
assert (ROOT / "dataset.yaml").exists(), "No está /content/dataset/dataset.yaml: el zip tiene que contener la carpeta dataset/ en la raíz."
for split in ("train", "val", "test"):
    imgs = list((ROOT / "images" / split).glob("*.jpg"))
    lbls = list((ROOT / "labels" / split).glob("*.txt"))
    print(f"{split}: {len(imgs)} imágenes, {len(lbls)} etiquetas")

## 4. Corregir la ruta del `dataset.yaml`

`build_dataset.py` escribe la ruta de la PC (por ejemplo `C:/Users/...`). En Colab tiene que apuntar a `/content/dataset`.

In [ ]:
import yaml

DATA_YAML = ROOT / "dataset.yaml"
d = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
print("ruta original:", d["path"])
d["path"] = str(ROOT)
DATA_YAML.write_text(yaml.safe_dump(d, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

## 5. Instalar Ultralytics

In [ ]:
!pip install -q -U ultralytics
import ultralytics
ultralytics.checks()

## 6. Entrenar

Los parámetros son los de `train:` en `ml/config.yaml`. Solo cambian `device` (GPU) y `batch`, que en la PC eran `cpu` y `8`. Si se cambia algo en `config.yaml`, cambiarlo también acá.

In [ ]:
from ultralytics import YOLO

TRAIN = {
    "epochs": 60,
    "patience": 20,
    "imgsz": 640,
    "batch": 32,        # en config.yaml: 8 (CPU)
    "workers": 2,
    "device": 0,        # en config.yaml: cpu
    # Aumentaciones pensadas para una botella de frente: sin flip vertical, giro leve.
    "fliplr": 0.5,
    "flipud": 0.0,
    "degrees": 5.0,
    "translate": 0.08,
    "scale": 0.3,
    "hsv_h": 0.01,      # bajo para no confundir la tapa roja (Clásico) con la verde (Stevia)
    "hsv_s": 0.5,
    "hsv_v": 0.4,
    "mosaic": 0.5,
    "close_mosaic": 15,
}

model = YOLO("yolo26n.pt")  # nano, NMS-free; alternativa: yolo11n.pt
model.train(
    data=str(DATA_YAML),
    project="/content/runs",
    name="detect",
    exist_ok=True,
    plots=True,
    **TRAIN,
)

## 7. Ver resultados

In [ ]:
from IPython.display import Image, display

RUN = Path("/content/runs/detect")
for name in ("results.png", "confusion_matrix.png", "val_batch0_pred.jpg"):
    if (RUN / name).exists():
        print(name)
        display(Image(filename=str(RUN / name), width=900))

## 8. Descargar `best.pt`

Copiarlo en la PC como `ml/work/best.pt` y seguir con la evaluación y la exportación:

```powershell
cd ml; .venv\Scripts\python pipeline.py --from evaluate
```

In [ ]:
BEST = RUN / "weights" / "best.pt"
assert BEST.exists(), "No está best.pt: revisar que el entrenamiento haya terminado."
files.download(str(BEST))

In [ ]:
# Opcional: guardar la corrida completa en Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r /content/runs "/content/drive/MyDrive/centinela_runs"